# 0. Colab Runner (max-steps 조정 버전)

`formation_seq.py + comm_env.py + comm_train.py + comm_eval.py` 실행용 Colab 노트북

먼저 Colab 메뉴에서 `런타임 → 런타임 유형 변경 → GPU`를 선택하여 진행해야함

---
**속도 메모 (박근희)**: 프로파일링 결과 학습 시간의 약 70%가 torchrl PettingZooWrapper의 step 변환에 쓰여 코드로 wall-clock을 줄이긴 어려움. 다만 단계당 max-steps를 짧게 두면 episode가 빨리 끝나 reset이 잦아져 **같은 frame 수로 학습 진행도가 빨라짐**. GROUND,D,G(3모양) 기준 max-steps 220→120에서 동일 frame 기준 reward +2.327 → +3.297 (약 1.4배) 확인. 본 노트북은 1~2단계 학습/평가 셀의 max-steps를 120으로 통일함. (다단계 A,B,C,D 셀은 모양 수가 많아 step이 더 필요하므로 기존값 유지 — 별도 검증 전까지 줄이지 않음.)


# 1. 기존 폴더 제거 후 GitHub clone

최신 patch 덮어쓰기

파일이 없다면 Colab 왼쪽 파일창에 따로 업로드한 뒤 실행

In [ ]:

!rm -rf RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git
%cd RL-2026s1-tp

# JSY branch로 이동
!git fetch origin
!git switch JSY
!git pull origin JSY

## 2. Google Drive 저장 경로 추가



In [ ]:

from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/drone_results/gifs
!mkdir -p /content/drive/MyDrive/drone_results/evals

## 3. 패키지 설치

In [ ]:
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

## 4. 파일 확인 및 smoke test

In [ ]:
%cd /content/RL-2026s1-tp

import inspect
import comm_env

print("comm_env file:", comm_env.__file__)
print("ShapeFormationEnv signature:")
print(inspect.signature(comm_env.ShapeFormationEnv))

## 5. 빠른 학습 테스트

정상 실행 확인용. \
학습 성능은 추후 확인이 필요하며, reward의 정상적인 상승 수준만 확인 \
(success의 경우 상승하지 않을 수 있음)

In [ ]:
%cd /content/RL-2026s1-tp
!python comm_train.py \
  --grid-size 25 \
  --n-agents 20 \
  --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.2 \
  --hover-penalty 0.05 \
  --shaping-coef 0.3 \
  --total-frames 8192 \
  --frames-per-batch 1024 \
  --minibatch-size 256 \
  --ppo-epochs 5 \
  --lr 2e-4 \
  --ent-coef 0.015 \
  --ckpt-every 4 \
  --save-dir checkpoints_ground_x_c100_fast \
  --tb-logdir runs_ground_x_c100_fast

## 6. 본 학습: GROUND → A

In [ ]:
!python comm_train.py\
  --grid-size 25 \
  --n-agents 20 \
  --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 \
  --shaping-coef 0.5 \
  --total-frames 1048576 \
  --frames-per-batch 4096 \
  --minibatch-size 256 \
  --ppo-epochs 3 \
  --lr 5e-4 \
  --ent-coef 0.03 \
  --ckpt-every 5 \
  --save-dir checkpoints_ground_x_c120 \
  --tb-logdir runs_ground_x_c120

본 학습 이후 chpt에서 추가 실행하기 위한 코드

In [ ]:
!python comm_train.py \
  --grid-size 25 \
  --n-agents 20 \
  --max-steps 120 \
  --shapes GROUND,X \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.2 \
  --hover-penalty 0.03 \
  --shaping-coef 0.3 \
  --load-ckpt checkpoints_ground_x_c120/ckpt_170.pt \
  --total-frames 262144 \
  --frames-per-batch 8192 \
  --minibatch-size 1024 \
  --ppo-epochs 2 \
  --lr 5e-5 \
  --ent-coef 0.003 \
  --clip-eps 0.05 \
  --ckpt-every 5 \
  --save-dir checkpoints_ground_x_c120_from_ckpt170_stable \
  --tb-logdir runs_ground_x_c120_from_ckpt170_stable

## 7. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs_ground_x

## 8. 평가 및 GIF 저장

`ckpt_30.pt` 부분은 실제 생성된 checkpoint 번호로 바꿀 것

In [ ]:
!python comm_eval.py \
  --ckpt checkpoints_ground_x_c100_fast/ckpt_8.pt \
  --grid-size 25 \
  --n-agents 20 \
  --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 100.0 \
  --greedy \
  --n-episodes 1 \
  --save-gif demo_ground_x_ckpt8.gif \
  --out eval_ground_x_ckpt8.txt

## 9. 다단계 전환 예시

GROUND→X가 먼저 어느 정도 성공하면 그 다음 실행

> ⚠️ 이 다단계 셀의 max-steps는 아직 속도/학습 검증 전이라 기존값을 유지함. 모양 수가 많아 짧게 줄이면 학습이 안 될 수 있음.

In [ ]:
!python comm_train.py\
  --grid-size 25 \
  --n-agents 20 \
  --max-steps 200 \
  --shapes GROUND,A,B,C,D \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 \
  --shaping-coef 0.5 \
  --total-frames 1048576 \
  --frames-per-batch 4096 \
  --minibatch-size 256 \
  --ppo-epochs 3 \
  --lr 5e-4 \
  --ent-coef 0.03 \
  --ckpt-every 5 \
  --save-dir checkpoints_ground_ABCD_s200 \
  --tb-logdir runs_ground_ABCD_s200

## 10. 멀티 시퀀스의 평가 및 GIF 저장

`ckpt_100.pt` 부분은 실제 생성된 checkpoint 번호로 바꿀 것

In [ ]:
!python comm_eval.py \
  --ckpt checkpoints_ground_ABCD_s200/ckpt_5.pt \
  --grid-size 25 \
  --n-agents 20 \
  --max-steps 200 \
  --shapes GROUND,A,B,C,D \
  --completion-reward 100.0 \
  --greedy \
  --n-episodes 1 \
  --save-gif demo_ground_ABCD_s200.gif \
  --out eval_ground_ABCD_s200.txt